In [2]:
import sqlite3

def setup_database():
    conn = sqlite3.connect('university.db')
    cursor = conn.cursor()

    # 1. Create Tables
    cursor.execute("DROP TABLE IF EXISTS enrollments")
    cursor.execute("DROP TABLE IF EXISTS students")
    cursor.execute("DROP TABLE IF EXISTS courses")

    cursor.execute("""
        CREATE TABLE students (
            student_id INTEGER PRIMARY KEY AUTOINCREMENT,
            full_name TEXT NOT NULL
        )
    """)

    cursor.execute("""
        CREATE TABLE courses (
            course_id INTEGER PRIMARY KEY AUTOINCREMENT,
            course_name TEXT NOT NULL
        )
    """)

    # Junction Table for Many-to-Many Relationship
    cursor.execute("""
        CREATE TABLE enrollments (
            student_id INTEGER,
            course_id INTEGER,
            PRIMARY KEY (student_id, course_id),
            FOREIGN KEY (student_id) REFERENCES students(student_id),
            FOREIGN KEY (course_id) REFERENCES courses(course_id)
        )
    """)

    # 2. Predefined Courses (At least 5)
    courses = [("BS ECE",), ("BS CE",), ("BS EE",), 
               ("BS CPE",), ("BS ME",)]
    cursor.executemany("INSERT INTO courses (course_name) VALUES (?)", courses)

    # 3. Sample Students (At least 10)
    students = [("Angelie Estrella",), ("Prince Rivera",), ("Peter Cedric",), 
                ("John Lordan",), ("Alden Paracuelles",), ("Farah Abdallah",), 
                ("Lyra Erguiza",), ("Raym Geronimo",), ("Aira Bartolome",), ("Jodel Heart",)]
    cursor.executemany("INSERT INTO students (full_name) VALUES (?)", students)

    # 4. Sample Enrollments (At least 15)
    # Mapping student IDs to course IDs
    enrollments = [(1,1), (1,2), (2,1), (2,3), (3,1), (3,4), (3,5), 
                   (4,2), (4,3), (5,5), (6,1), (7,2), (8,3), (9,4), (10,5)]
    cursor.executemany("INSERT OR IGNORE INTO enrollments VALUES (?, ?)", enrollments)

    conn.commit()
    return conn

def enroll_student(conn, student_id, course_id):
    cursor = conn.cursor()
    try:
        # Parameterized query to check for duplicates and protect against injection
        cursor.execute("INSERT INTO enrollments (student_id, course_id) VALUES (?, ?)", (student_id, course_id))
        conn.commit()
        print("Success: Enrollment completed.")
    except sqlite3.IntegrityError:
        print("WARNING: This student is already enrolled in this course.")

def list_student_courses(conn, student_name):
    cursor = conn.cursor()
    cursor.execute("""
        SELECT c.course_name 
        FROM courses c
        JOIN enrollments e ON c.course_id = e.course_id
        JOIN students s ON e.student_id = s.student_id
        WHERE s.full_name LIKE ?
    """, (f"%{student_name}%",))
    
    rows = cursor.fetchall()
    print(f"\nCourses for {student_name}:")
    for row in rows:
        print(f"- {row[0]}")

def list_course_students(conn, course_id):
    cursor = conn.cursor()
    cursor.execute("""
        SELECT s.full_name 
        FROM students s
        JOIN enrollments e ON s.student_id = e.student_id
        WHERE e.course_id = ?
    """, (course_id,))
    
    rows = cursor.fetchall()
    print(f"\nStudents enrolled in Course ID {course_id}:")
    for row in rows:
        print(f"- {row[0]}")

# --- Main Program Execution ---
conn = setup_database()

# Demonstrate Output 1: List all courses a student is enrolled in
list_student_courses(conn, "Peter Cedric")

# Demonstrate Output 2: List all students in a specific course (e.g., Calculus I - ID 1)
list_course_students(conn, 1)

# Demonstrate Output 3: Warning for duplicate enrollment
print("\nAttempting duplicate enrollment for Angelie Estrella in BS EE...")
enroll_student(conn, 1, 1) 

conn.close()


Courses for Peter Cedric:
- BS ECE
- BS CPE
- BS ME

Students enrolled in Course ID 1:
- Angelie Estrella
- Prince Rivera
- Peter Cedric
- Farah Abdallah

Attempting duplicate enrollment for Angelie Estrella in BS EE...


In [11]:
# Exercise 2

import sqlite3
from datetime import datetime

# ==========================================
# DATABASE INITIALIZATION
# ==========================================

def init_attendance_db():
    conn = sqlite3.connect("attendance_tracker.db")
    cursor = conn.cursor()
    cursor.execute("PRAGMA foreign_keys = ON;")

    # 1. Students Table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS students (
        student_id INTEGER PRIMARY KEY AUTOINCREMENT,
        full_name TEXT NOT NULL
    );
    """)

    # 2. Attendance Records Table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS attendance (
        attendance_id INTEGER PRIMARY KEY AUTOINCREMENT,
        student_id INTEGER,
        attendance_date TEXT NOT NULL,
        status TEXT NOT NULL CHECK (status IN ('Present', 'Absent')),
        FOREIGN KEY (student_id) REFERENCES students(student_id) ON DELETE CASCADE
    );
    """)
    conn.commit()
    return conn


def seed_students(conn):
    """Seed sample students if the table is empty."""
    cursor = conn.cursor()
    cursor.execute("SELECT COUNT(*) FROM students")
    if cursor.fetchone()[0] == 0:
        sample_students = [
            ("Peter Cedric",), ("Angelie Estrella",), ("Prince Rivera",), 
            ("John Floyd",), ("Alden Paracuelles",)
        ]
        cursor.executemany("INSERT INTO students (full_name) VALUES (?)", sample_students)
        conn.commit()


# ==========================================
# ATTENDANCE OPERATIONS
# ==========================================

def mark_attendance(conn):
    """Inserts a new attendance record for a student."""
    cursor = conn.cursor()
    
    # List students
    cursor.execute("SELECT student_id, full_name FROM students")
    students = cursor.fetchall()
    print("\n--- Student List ---")
    for s in students:
        print(f"[{s[0]}] {s[1]}")

    try:
        student_id = int(input("\nSelect Student ID: "))
    except ValueError:
        print("Invalid input.")
        return

    # Validate student
    cursor.execute("SELECT full_name FROM students WHERE student_id = ?", (student_id,))
    if not cursor.fetchone():
        print("Error: Student ID not found.")
        return

    # Input date
    date_input = input("Enter Date (YYYY-MM-DD) or press Enter for today: ").strip()
    if not date_input:
        date_input = datetime.today().strftime('%Y-%m-%d')
    else:
        try:
            # Validate format
            datetime.strptime(date_input, '%Y-%m-%d')
        except ValueError:
            print("Incorrect date format. Use YYYY-MM-DD.")
            return

    # Input status
    status = input("Enter Status (P for Present, A for Absent): ").strip().upper()
    if status == 'P':
        status_text = 'Present'
    elif status == 'A':
        status_text = 'Absent'
    else:
        print("Invalid status choice.")
        return

    # Insert using SQLite DATE() function processing
    try:
        cursor.execute("""
        INSERT INTO attendance (student_id, attendance_date, status)
        VALUES (?, DATE(?), ?)
        """, (student_id, date_input, status_text))
        conn.commit()
        print(f"Success: Marked {status_text} for student ID {student_id} on {date_input}.")
    except sqlite3.Error as e:
        print(f"Error logging attendance: {e}")


def view_attendance_by_student(conn):
    """Filters attendance records by student ID."""
    cursor = conn.cursor()
    try:
        student_id = int(input("Enter Student ID to view records: "))
    except ValueError:
        print("Invalid entry.")
        return

    cursor.execute("""
    SELECT s.full_name, a.attendance_date, a.status
    FROM attendance a
    JOIN students s ON a.student_id = s.student_id
    WHERE a.student_id = ?
    ORDER BY a.attendance_date DESC
    """, (student_id,))
    
    records = cursor.fetchall()
    if records:
        print(f"\n--- Attendance History for {records[0][0]} ---")
        for r in records:
            print(f"Date: {r[1]} | Status: {r[2]}")
    else:
        print("No attendance records found for this student.")


def view_attendance_by_date(conn):
    """Filters attendance records by a specific date."""
    date_input = input("Enter Date (YYYY-MM-DD) to filter: ").strip()
    try:
        datetime.strptime(date_input, '%Y-%m-%d')
    except ValueError:
        print("Incorrect date format. Use YYYY-MM-DD.")
        return

    cursor = conn.cursor()
    cursor.execute("""
    SELECT s.full_name, a.status
    FROM attendance a
    JOIN students s ON a.student_id = s.student_id
    WHERE a.attendance_date = DATE(?)
    """, (date_input,))
    
    records = cursor.fetchall()
    if records:
        print(f"\n--- Attendance Records for {date_input} ---")
        for r in records:
            print(f"Student: {r[0]:<18} | Status: {r[1]}")
    else:
        print(f"No records found for the date {date_input}.")


# ==========================================
# MAIN INTERFACE
# ==========================================

def main():
    conn = init_attendance_db()
    seed_students(conn)

    while True:
        print("\n===========================================")
        print("        ATTENDANCE RECORDING SYSTEM        ")
        print("===========================================")
        print("1. Mark Attendance")
        print("2. View Records by Student ID")
        print("3. View Records by Date")
        print("4. Exit")
        
        choice = input("\nEnter choice (1-4): ").strip()
        
        if choice == "1":
            mark_attendance(conn)
        elif choice == "2":
            view_attendance_by_student(conn)
        elif choice == "3":
            view_attendance_by_date(conn)
        elif choice == "4":
            conn.close()
            print("Attendance program closed. Goodbye!")
            break
        else:
            print("Invalid choice. Try again.")

if __name__ == "__main__":
    main()


        ATTENDANCE RECORDING SYSTEM        
1. Mark Attendance
2. View Records by Student ID
3. View Records by Date
4. Exit



--- Student List ---
[1] Peter Cedric
[2] Angelie Estrella
[3] Prince Rivera
[4] John Floyd
[5] Alden Paracuelles
Success: Marked Present for student ID 1 on 2026-05-05.

        ATTENDANCE RECORDING SYSTEM        
1. Mark Attendance
2. View Records by Student ID
3. View Records by Date
4. Exit
Attendance program closed. Goodbye!
